# theoryminer — installation test in Google Colab

This notebook checks that `theoryminer` 0.2.0 installs from PyPI with pip, and that each public function
works: from PDF to sentences, DOIs, references, downloads, relations, groups, names and the causal map.
Run the cells from top to bottom. Each cell ends with a check. A check that fails stops the cell with an
`AssertionError`. At the end, save the notebook, so that the outputs are stored in the file.

The notebook needs an internet connection: for pip, the sample papers, the models and the downloads.

## 1. Install from PyPI, with the download extra

In [ ]:
%pip install -q "theoryminer[download]"

## 2. The environment

In [ ]:
import os
import sys
import torch
import theoryminer

package_dir = os.path.dirname(theoryminer.__file__)
print("python     :", sys.version.split()[0])
print("torch      :", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("theoryminer:", theoryminer.__version__, "|", package_dir)
print("data files :", sorted(os.listdir(os.path.join(package_dir, "data"))))

version = tuple(int(part) for part in theoryminer.__version__.split(".")[:2])
assert version >= (0, 2), f"expected version 0.2.0 or newer, got {theoryminer.__version__}"
DATA_FILES = ["ELSST_R5.rdf.gz", "ELSST_LICENSE.md", "causal_map.html", "cytoscape.min.js",
              "layout-base.js", "cose-base.js", "cytoscape-fcose.js"]
for name in DATA_FILES:
    assert os.path.isfile(os.path.join(package_dir, "data", name)), f"{name} is missing from the package"
PUBLIC_NAMES = ["harvest", "extract_dois", "extract_references", "download_papers", "causenet",
                "standardize_constructs", "standardize_groups", "pairwise_grouping", "clusterer",
                "label_groups", "causal_map", "draw_map", "save_map"]
for name in PUBLIC_NAMES:
    assert hasattr(theoryminer, name), f"theoryminer.{name} is missing"
print("check passed: version", theoryminer.__version__, "| 7 data files | 13 public functions")

## 3. Download the sample papers

In [ ]:
import json
import urllib.request

REPO = "rasoulnorouzi/theoryminer"


def download_sample_files(folder="sample_files"):
    """Download every file of sample_files/ from GitHub into the folder."""
    listing_url = f"https://api.github.com/repos/{REPO}/contents/sample_files"
    with urllib.request.urlopen(listing_url) as response:
        listing = json.load(response)
    os.makedirs(folder, exist_ok=True)
    for item in listing:
        urllib.request.urlretrieve(item["download_url"], os.path.join(folder, item["name"]))
    return listing


listing = download_sample_files()
print(len(listing), "files:", sorted(item["name"] for item in listing))
assert len(listing) == 8, "expected 7 PDF files and SOURCES.md"
print("check passed: 8 files downloaded")

## 4. `harvest()`: one PDF, a folder, and bad files

In [ ]:
import shutil
from theoryminer import harvest

data = harvest("sample_files")
sent_ids = [s["sent_id"] for s in data["sentences"]]
doc_ids = {s["doc_id"] for s in data["sentences"]}

assert len(doc_ids) == 7, "expected 7 documents"
front = [s for s in data["sentences"] if s["flag"] == "front_matter"]
assert len(front) > 0, "expected some sidebar sentences with the flag front_matter"
assert len(sent_ids) == len(set(sent_ids)), "sent_id is not unique"
print()
print("check passed:", len(sent_ids), "sentences from", len(doc_ids), "documents; every sent_id is unique;",
      len(front), "flagged front_matter")

reasons = {row["drop_reason"] for row in data["dropped"]}
assert "licence_text" in reasons, "the licence text of the journals is not dropped"
print("check passed: the licence text is dropped with the reason licence_text")

In [ ]:
# A folder with one good PDF and three bad files.
os.makedirs("bad_folder", exist_ok=True)
shutil.copy("sample_files/barrech_2018_job_insecurity_health.pdf", "bad_folder/good.pdf")
with open("bad_folder/notes.txt", "w") as fh:
    fh.write("A text file, not a PDF.")
with open("bad_folder/webpage.pdf", "w") as fh:
    fh.write("<!DOCTYPE html><html><body>Access denied</body></html>")
with open("bad_folder/empty.pdf", "w") as fh:
    fh.write("")

print("bad_folder holds", len(os.listdir("bad_folder")), "files:", sorted(os.listdir("bad_folder")))
print()
mixed = harvest("bad_folder", cache=False)
skipped = {}
for row in mixed["dropped"]:
    if row["page"] == 0:
        skipped[row["doc_id"]] = row["drop_reason"]
print()
print("skipped files:", skipped)
assert skipped == {"notes": "not_pdf", "webpage": "unreadable_pdf", "empty": "unreadable_pdf"}
print("check passed: the bad files are logged, and the good file is read")

## 4b. `extract_dois()`: the DOI and the title of each paper

In [ ]:
from theoryminer import extract_dois

ANSWER_KEY = {  # read by hand from page 1 of each PDF
    "alrabai_2021_autonomy_supportive_teaching.pdf": "10.3389/fpsyg.2021.728657",
    "barrech_2018_job_insecurity_health.pdf": "10.1186/s12889-018-5621-4",
    "friesinger_2025_discrimination_health.pdf": "10.1186/s12889-025-23888-6",
    "gordesli_2024_social_media_mental_health.pdf": "10.1371/journal.pone.0316365",
    "pelikan_2021_needs_intrinsic_motivation.pdf": "10.1371/journal.pone.0257346",
    "schutz_2025_loneliness_social_support.pdf": "10.1186/s12889-025-23247-5",
    "stephany_2017_income_inequality_trust.pdf": "10.1007/s11205-016-1460-9",
}
rows = extract_dois("sample_files", save="outputs")
found = {}
for row in rows:
    if row["status"] == "found":
        found[row["file"]] = row["doi"]
assert found == ANSWER_KEY, "a DOI differs from the answer key"
assert os.path.isfile("outputs/dois.csv"), "the CSV file is missing"

bad = extract_dois("bad_folder", cache=False)
status = {}
for row in bad:
    status[row["file"]] = row["status"]
assert status == {"good.pdf": "found", "notes.txt": "not_pdf",
                  "webpage.pdf": "unreadable_pdf", "empty.pdf": "unreadable_pdf"}
print()
print("check passed: 7 of 7 DOIs are correct, the CSV file exists, and the bad files get a status")

titles = [row["title"] for row in rows if row["status"] == "found"]
assert len(titles) == 7 and all(titles), "a sample paper has no title"
print("check passed: 7 of 7 papers have a title, for example:", titles[-1])

## 4c. `extract_references()`: the DOIs in the references

In [ ]:
from theoryminer import extract_references

refs = extract_references("sample_files", save="outputs")
dois = [row["doi"] for row in refs]
assert len(refs) > 200, "expected more than 200 reference DOIs in the 7 sample papers"
assert "10.1177/14034948221117970" in dois, "a DOI that the PDF cuts at a line end was not repaired"
assert refs[0]["doi"] == "10.1207/s15327965pli1104_01" and refs[0]["n_seeds"] == 2, "the most cited DOI is wrong"
assert os.path.isfile("outputs/references.csv"), "the CSV file is missing"
print("check passed:", len(refs), "reference DOIs; the cut DOIs are repaired; the most cited DOI is right")

## 4d. `download_papers()`: the CC BY papers that one seed paper cites

Not every paper can be downloaded: many publishers block robots. The check looks at the statuses and
the files, not at a number of downloads.

In [ ]:
import importlib.util
from theoryminer import download_papers

assert importlib.util.find_spec("doi_downloader") is not None, "the extra [download] did not install tmsr-doi-downloader"
seed_refs = extract_references("sample_files/schutz_2025_loneliness_social_support.pdf")
report = download_papers(seed_refs, "downloads", licences=["cc-by"], save="outputs")
statuses = {row["status"] for row in report}
assert statuses <= {"downloaded", "already_there", "skipped_licence", "not_found", "error"}, statuses
assert len(report) == len(seed_refs), "expected one status row per DOI"
for row in report:
    if row["status"] == "downloaded":
        with open(os.path.join("downloads", row["file"]), "rb") as fh:
            assert fh.read(5) == b"%PDF-", f"{row['file']} is not a PDF"
        assert row["licence"] == "cc-by", f"{row['doi']} is not CC BY"
assert not os.path.exists("database.db"), "the downloader wrote its cache into the working folder"
counts = {status: sum(r["status"] == status for r in report) for status in sorted(statuses)}
print("check passed:", counts)

## 5. `causenet()`: find the cause-effect pairs

In [ ]:
import inspect
from theoryminer import causenet

defaults = inspect.signature(causenet).parameters
assert defaults["decision"].default == "span_only" and defaults["avoid_ambiguous"].default is True

toy = causenet(["Smoking causes lung cancer and heart disease",
                "Exercise improves physical health and mental well-being",
                "This leads to higher stress among employees",
                "This is a non-causal sentence about weather"])
print(toy[:2])
assert len(toy) >= 2, "the model found fewer relations than expected"
assert all(r["cause"].lower() != "this" for r in toy), "a relation with the pointer span 'this' was not skipped"
print("check passed: the model loads, finds relations, and skips the pointer span 'this'")

In [ ]:
relations = causenet(data["sentences"])
rel_ids = [r["rel_id"] for r in relations]
assert len(rel_ids) == len(set(rel_ids)), "rel_id is not unique"
assert all(r["cause"].strip() and r["effect"].strip() for r in relations), "a relation has an empty span"
print()
print("check passed:", len(relations), "relations; every rel_id is unique; no empty span")
for r in relations[:5]:
    print(f"  [{r['rel_id']} p{r['page']}] {r['cause']!r} -> {r['effect']!r}")

## 6. The harmonizer: ELSST, grouping, clustering, names

In [ ]:
from theoryminer.harmonizer import (
    standardize_constructs, standardize_groups, pairwise_grouping, clusterer, label_groups)

assert inspect.signature(standardize_constructs).parameters["leaves_only"].default is False
assert inspect.signature(pairwise_grouping).parameters["linkage"].default == "average"
assert inspect.signature(label_groups).parameters["by"].default == "keywords"

probe = ["loneliness", "social trust", "unemployment", "intrinsic motivation"]
matches = standardize_constructs(probe, top=3)
print()
for span in probe:
    best = matches[span][0]
    print(f"  {span:<22} -> {best['leaf']} ({best['score']:.2f})")
assert all(len(matches[span]) == 3 for span in probe)
print("check passed: the new defaults are set; the packaged ELSST file loads, and the spans get concepts")

In [ ]:
import collections

groups = pairwise_grouping(relations)                             # average linkage
groups_single = pairwise_grouping(relations, linkage="single")    # the method before 0.2.0
clusters = clusterer(relations)
clusters_umap = clusterer(relations, umap=dict(n_components=5, n_neighbors=15, min_dist=0.0))
named = standardize_groups(clusters, name_by="mean_sim", top=3)
labelled = label_groups(clusters)                                 # the TF-IDF keywords


def largest(groups):
    return max(collections.Counter(g["group"] for g in groups.values()).values())


spans = {r["cause"] for r in relations} | {r["effect"] for r in relations}
assert set(groups) == spans and set(clusters) == spans, "a span is missing from a result"
assert largest(groups) < largest(groups_single), "average linkage should stop the chaining of single linkage"
assert all(" | " not in row["name"] for row in labelled.values()), "the labels are not keywords only"
print()
print(f"check passed: every span has a group, a cluster and a name; largest group: "
      f"{largest(groups)} (average) against {largest(groups_single)} (single)")

In [ ]:
named

## 6b. The causal map: `causal_map()`, `draw_map()`, `save_map()`

In [ ]:
from theoryminer import causal_map, draw_map, save_map

# Clustering, named by the TF-IDF keywords. The DOIs and titles come from extract_dois() (section 4b).
cmap = causal_map(relations, labelled)
assert cmap["settings"]["min_sentences"] == 2, "the default filter is not min_sentences=2"
assert cmap["edges"], "the map has no shown edge"
all_edges = cmap["edges"] + cmap["hidden"]
assert all(e["n_sentences"] <= e["n_relations"] for e in all_edges), "an edge counts more sentences than relations"
draw_map(cmap, papers=rows)                      # the interactive page shows below this cell
paths = save_map(cmap, "outputs/causal_map")
assert all(os.path.isfile(p) for p in paths), "save_map() did not write its four files"
print("check passed:", len(cmap["edges"]), "edges shown,", len(cmap["hidden"]), "hidden; 4 files written")

In [ ]:
# The same clusters, named by ELSST concepts: clusters with one best concept share a node.
cmap_elsst = causal_map(relations, named)
draw_map(cmap_elsst, papers=rows, save="outputs/causal_map_elsst.html", title="Clusters named by ELSST")
assert os.path.isfile("outputs/causal_map_elsst.html"), "draw_map() did not write the page"
print("check passed: the ELSST-named map has", len(cmap_elsst["edges"]), "edges shown")

## 6c. A small snowball round: the downloaded papers are the next input

In [ ]:
pdfs = [name for name in os.listdir("downloads") if name.endswith(".pdf")]
if not pdfs:
    print("no PDF was downloaded in section 4d; this check is skipped")
else:
    round_2 = harvest("downloads")
    next_refs = extract_references("downloads")
    assert round_2["sentences"], "the downloaded papers gave no sentence"
    print("check passed:", len(pdfs), "downloaded papers ->", len(round_2["sentences"]), "sentences and",
          len(next_refs), "reference DOIs for the next round")

## 7. Summary

In [ ]:
print("version    :", theoryminer.__version__)
print("sentences  :", len(data["sentences"]))
print("references :", len(refs))
print("downloads  :", sum(r["status"] == "downloaded" for r in report), "of", len(report))
print("relations  :", len(relations))
print("spans      :", len(spans))
print("clusters   :", len({g["group"] for g in clusters.values() if g["group"] != -1}))
print("map        :", len(cmap["edges"]), "edges shown")
print("ALL CHECKS PASSED")